In [1]:
import h5py
import numpy as np
import torch
from emle.models import EMLE
from emle.train._utils import pad_to_max

ANGSTROM_TO_BOHR = 1.8897259886
HARTREE_TO_KJ_MOL = 2625.5

cuequivariance or cuequivariance_torch is not available. Cuequivariance acceleration will be disabled.


In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dtype = torch.float32

### Load Data

In [3]:
# This corresponds to the DES370K dataset filtered to:
# - include only neutral molecuiels
# - include only molecules with elements H, C, N, O, S
# - not include molecules for which interchange failed (viz., [H][H])
DES_FILE = "des_filtered.h5"

In [ ]:
data = h5py.File(DES_FILE, "r")

z = data["atomic_numbers"][:]
z_mm = data["atomic_numbers_mm"][:]
xyz_qm = data["xyz_qm"][:]
xyz_mm = data["xyz_mm"][:]
charges_mm = data["charges_mm"][:]
e_disp = data["e_disp"][:]
m1 = data["xdm_M1_sq"][:]
m2 = data["xdm_M2_sq"][:]
m3 = data["xdm_M3_sq"][:]
m1_mm = data["xdm_M1_sq_mm"][:]
m2_mm = data["xdm_M2_sq_mm"][:]
m3_mm = data["xdm_M3_sq_mm"][:]

# Check for nans in e_disp
mask = ~np.isnan(e_disp)
e_disp = e_disp[mask]

# Pad arrays to max sizes
z = pad_to_max(z).to(device=device, dtype=torch.long)[mask]
z_mm = pad_to_max(z_mm).to(device=device, dtype=torch.long)[mask]
xyz_qm = pad_to_max(xyz_qm).to(device=device, dtype=dtype)[mask]
xyz_mm = pad_to_max(xyz_mm).to(device=device, dtype=dtype)[mask]
charges_mm = pad_to_max(charges_mm).to(device=device, dtype=dtype)[mask]
m1 = pad_to_max(m1).to(device=device, dtype=dtype)[mask]
m2 = pad_to_max(m2).to(device=device, dtype=dtype)[mask]
m3 = pad_to_max(m3).to(device=device, dtype=dtype)[mask]
m1_mm = pad_to_max(m1_mm).to(device=device, dtype=dtype)[mask]
m2_mm = pad_to_max(m2_mm).to(device=device, dtype=dtype)[mask]
m3_mm = pad_to_max(m3_mm).to(device=device, dtype=dtype)[mask]
q_mol = torch.zeros(z.shape[0], device=device, dtype=dtype)

### Create EMLE model and calculate EMLE parameters

In [ ]:
emle = EMLE(
    model="../../emle_models/ligand_patched_static_only.mat",
    device=device,
    dtype=dtype,
    alpha_mode="reference",
)
emle_base = emle._emle_base

In [ ]:
# Calculate reference values in batches
batch_size = 512
n_configs = xyz_mm.shape[0]
total_e_lj = []
A_thole = []
c6_qm = []
s_qm = []
sigma_scale_qm = []
q_val_qm_list = []
with torch.no_grad():
    for start in range(0, n_configs, batch_size):
        print(f"Processing batch {start}/{n_configs}")
        end = min(start + batch_size, n_configs)
        z_batch = z[start:end]
        charges_mm_batch = charges_mm[start:end]
        xyz_qm_batch = xyz_qm[start:end]
        xyz_mm_batch = xyz_mm[start:end]
        q_mol_batch = q_mol[start:end]

        # Zero-out elements outside batch
        e_static, e_ind, _, _, e_lj_batch = emle.forward(
            z_batch, charges_mm_batch, xyz_qm_batch, xyz_mm_batch
        )

        s, q_core, q_val, A_thole_batch, c6, _, sigma_scale = emle_base.forward(
            z_batch, xyz_qm_batch, q_mol_batch, calc_A_thole=True, calc_c6=False
        )

        A_thole.append(A_thole_batch)
        total_e_lj.append(e_lj_batch)
        c6_qm.append(c6)
        s_qm.append(s)
        sigma_scale_qm.append(sigma_scale)
        q_val_qm_list.append(q_val)

e_lj_init = torch.cat(total_e_lj, dim=0)
A_thole = torch.cat(A_thole, dim=0)
s_qm = torch.cat(s_qm, dim=0)
sigma_scale_qm_init = torch.cat(sigma_scale_qm, dim=0)
q_val_qm = torch.cat(q_val_qm_list, dim=0)
alpha_qm = emle_base.get_isotropic_polarizabilities_thole(A_thole) * (z > 0)

Processing batch 0/251952
Processing batch 512/251952
Processing batch 1024/251952
Processing batch 1536/251952
Processing batch 2048/251952
Processing batch 2560/251952
Processing batch 3072/251952
Processing batch 3584/251952
Processing batch 4096/251952
Processing batch 4608/251952
Processing batch 5120/251952
Processing batch 5632/251952
Processing batch 6144/251952
Processing batch 6656/251952
Processing batch 7168/251952
Processing batch 7680/251952
Processing batch 8192/251952
Processing batch 8704/251952
Processing batch 9216/251952
Processing batch 9728/251952
Processing batch 10240/251952
Processing batch 10752/251952
Processing batch 11264/251952
Processing batch 11776/251952
Processing batch 12288/251952
Processing batch 12800/251952
Processing batch 13312/251952
Processing batch 13824/251952
Processing batch 14336/251952
Processing batch 14848/251952
Processing batch 15360/251952
Processing batch 15872/251952
Processing batch 16384/251952
Processing batch 16896/251952
Proc

In [ ]:
# Calculate MM polarizabilities
A_thole_mm = []
s_mm = []
q_val_mm_list = []
with torch.no_grad():
    for start in range(0, n_configs, batch_size):
        end = min(start + batch_size, n_configs)
        z_mm_batch = z_mm[start:end]
        xyz_mm_batch = xyz_mm[start:end]
        q_mol_batch = q_mol[start:end]

        s, q_core, q_val, A_thole_mm_batch, *_ = emle_base.forward(
            z_mm_batch, xyz_mm_batch, q_mol_batch, calc_A_thole=True, calc_c6=False
        )

        A_thole_mm.append(A_thole_mm_batch)
        s_mm.append(s)
        sigma_scale_qm.append(sigma_scale)
        q_val_mm_list.append(q_val)

A_thole_mm = torch.cat(A_thole_mm, dim=0)
alpha_mm = emle_base.get_isotropic_polarizabilities_thole(A_thole_mm) * (z_mm > 0)

In [ ]:
# QM data has shape (N_batch, N_QM, 1)
m1 = m1[:, :, None]
m2 = m2[:, :, None]
m3 = m3[:, :, None]
alpha_qm = alpha_qm[:, :, None]

# MM data has shape (N_batch, 1, N_MM)
m1_mm = m1_mm[:, None, :]
m2_mm = m2_mm[:, None, :]
m3_mm = m3_mm[:, None, :]
alpha_mm = alpha_mm[:, None, :]

### XDM Dispersion Energy

In [ ]:
def c6(alpha_i, alpha_j, m1_i, m1_j):
    """XDM C6 coefficient calculation for heteroatomic case."""
    return (alpha_i * alpha_j * m1_i * m1_j) / (alpha_j * m1_i + alpha_i * m1_j + 1e-16)


def c8(alpha_i, m1_i, m2_i, alpha_j, m1_j, m2_j):
    """XDM C8 coefficient calculation for heteroatomic case."""
    num = alpha_i * alpha_j * (m1_i * m2_j + m2_i * m1_j)
    denom = alpha_j * m1_i + alpha_i * m1_j + 1e-16
    return (3 / 2.0) * (num / denom)


def c10(alpha_i, m1_i, m2_i, m3_i, alpha_j, m1_j, m2_j, m3_j):
    """XDM C10 coefficient calculation for homoatomic case."""
    num_term1 = 2 * (alpha_i * alpha_j) * (m1_i * m3_j + m3_i * m1_j)
    num_term2 = (21 / 5.0) * (alpha_i * alpha_j) * (m2_i * m2_j)
    denom_term1 = alpha_j * m1_i + alpha_i * m1_j + 1e-16
    denom_term2 = alpha_j * m1_i + alpha_i * m1_j + 1e-16
    return (num_term1 / denom_term1) + (num_term2 / denom_term2)


def r_vdw(c6, c8, c10, a1=1.0, a2=0.0):
    """https://doi.org/10.1063/5.0207682"""
    r_crit = (1 / 3.0) * (
        (c8 / (c6 + 1e-16)) ** 0.5
        + (c10 / (c6 + 1e-16)) ** 0.25
        + (c10 / (c8 + 1e-16)) ** 0.5
    )
    return a1 * r_crit + a2

In [ ]:
# Calculate dispersion coefficients
c6_xdm = c6(alpha_qm, m1, alpha_mm, m1_mm)
c8_xdm = c8(alpha_qm, m1, m2, alpha_mm, m1_mm, m2_mm)
c10_xdm = c10(alpha_qm, m1, m2, m3, alpha_mm, m1_mm, m2_mm, m3_mm)

# Pairwise Monomer1-Monomer2 distances
r = torch.cdist(xyz_qm, xyz_mm) * ANGSTROM_TO_BOHR + 1e-16

# XDM dispersion energies
mask_mm = (z_mm > 0).unsqueeze(1)
mask_qm = (z > 0).unsqueeze(2)
mask = mask_mm & mask_qm
e_disp_xdm = (
    -(c6_xdm / (r**6) + c8_xdm / (r**8) + c10_xdm / (r**10)) * HARTREE_TO_KJ_MOL
)
e_disp_xdm = torch.where(
    mask, e_disp_xdm, torch.tensor(0.0, device=device, dtype=dtype)
)
e_disp_xdm = e_disp_xdm.sum(dim=(1, 2)).detach().cpu().numpy()
e_disp_xdm_init = e_disp_xdm.copy()

### Optimization Loop

In [ ]:
if isinstance(e_disp, torch.Tensor):
    e_disp_ref = e_disp.to(device=device, dtype=dtype).view(-1)
else:
    e_disp_ref = torch.tensor(e_disp, device=device, dtype=dtype).view(-1)

# Trainable parameters
a1 = torch.nn.Parameter(
    torch.tensor(1.0, device=device, dtype=dtype), requires_grad=True
)
a2 = torch.nn.Parameter(
    torch.tensor(1.0, device=device, dtype=dtype), requires_grad=True
)
optimizer = torch.optim.Adam([a1, a2], lr=1e-2)

# Calculate dispersion coefficients for training
c6_x = c6(alpha_qm, m1, alpha_mm, m1_mm)
c8_x = c8(alpha_qm, m1, m2, alpha_mm, m1_mm, m2_mm)
c10_x = c10(alpha_qm, m1, m2, m3, alpha_mm, m1_mm, m2_mm, m3_mm)

# Pairwise Monomer1-Monomer2 distances
r_mat = torch.cdist(xyz_qm, xyz_mm) * ANGSTROM_TO_BOHR + 1e-16

# Masks
mask_mm = (z_mm > 0).unsqueeze(1)
mask_qm = (z > 0).unsqueeze(2)
pair_mask = mask_mm & mask_qm

# Optimization loop
num_epochs = 3000
best = {"loss": float("inf"), "a1": None, "a2": None}
for epoch in range(1, num_epochs + 1):
    optimizer.zero_grad()
    r_vdw_x = r_vdw(c6_x, c8_x, c10_x, a1=a1, a2=a2)
    e_pred = (
        -(
            c6_x / (r_mat**6 + r_vdw_x**6)
            + c8_x / (r_mat**8 + r_vdw_x**8)
            + c10_x / (r_mat**10 + r_vdw_x**10)
        )
        * HARTREE_TO_KJ_MOL
    )
    e_pred = torch.where(
        pair_mask, e_pred, torch.tensor(0.0, device=device, dtype=dtype)
    )
    e_pred = e_pred.sum(dim=(1, 2))
    loss = (e_pred - e_disp_ref).pow(2).mean()

    loss.backward(retain_graph=True)
    optimizer.step()

    if loss.item() < best["loss"]:
        best = {"loss": loss.item(), "a1": a1.detach().item(), "a2": a2.detach().item()}

    if epoch % 100 == 0 or epoch == 1 or epoch == num_epochs:
        print(
            f"Epoch {epoch:4d} | Loss {loss.item():.6f} | a1={a1.item():.6f}, a2={a2.item():.6f}"
        )

print("Best:", best)

with torch.no_grad():
    r_vdw_x = r_vdw(c6_x, c8_x, c10_x, a1=a1, a2=a2)
    e_pred = (
        -(
            c6_x / (r_mat**6 + r_vdw_x**6)
            + c8_x / (r_mat**8 + r_vdw_x**8)
            + c10_x / (r_mat**10 + r_vdw_x**10)
        )
        * HARTREE_TO_KJ_MOL
    )
    e_pred = torch.where(
        pair_mask, e_pred, torch.tensor(0.0, device=device, dtype=dtype)
    ).sum(dim=(1, 2))

Epoch    1 | Loss 260.211365 | a1=0.990000, a2=0.990000
Epoch  100 | Loss 12.172335 | a1=0.699552, a2=0.705826
Epoch  200 | Loss 12.162371 | a1=0.697746, a2=0.711716
Epoch  300 | Loss 12.158483 | a1=0.696181, a2=0.719904
Epoch  400 | Loss 12.154285 | a1=0.694382, a2=0.729278
Epoch  500 | Loss 12.150041 | a1=0.692430, a2=0.739452
Epoch  600 | Loss 12.145942 | a1=0.690385, a2=0.750111
Epoch  700 | Loss 12.142130 | a1=0.688300, a2=0.760984
Epoch  800 | Loss 12.138691 | a1=0.686220, a2=0.771837
Epoch  900 | Loss 12.135678 | a1=0.684182, a2=0.782473
Epoch 1000 | Loss 12.133106 | a1=0.682218, a2=0.792723
Epoch 1100 | Loss 12.130967 | a1=0.680354, a2=0.802453
Epoch 1200 | Loss 12.129230 | a1=0.678611, a2=0.811555
Epoch 1300 | Loss 12.127856 | a1=0.677004, a2=0.819951
Epoch 1400 | Loss 12.126796 | a1=0.675542, a2=0.827590
Epoch 1500 | Loss 12.125996 | a1=0.674230, a2=0.834447
Epoch 1600 | Loss 12.125408 | a1=0.673068, a2=0.840517
Epoch 1700 | Loss 12.124989 | a1=0.672054, a2=0.845817
Epoch 180

In [ ]:
data = {
    "e_disp_ref": e_disp_ref.cpu().numpy(),
    "e_disp_xdm_init": torch.tensor(e_disp_xdm_init, device=device, dtype=dtype).cpu().numpy(),
    "e_disp_xdm_final": e_pred.cpu().numpy(),
    "a1": a1.detach().item(),
    "a2": a2.detach().item(),
}

# Save as a compressed .npz file
np.savez_compressed("fit_results.npz", **data)